# Task 3 · Cleaning Data (Easy / Beginner Version)

**Objective:** This notebook cleans a "messy" (dirty / inconsistent) dataset step by step, so it becomes ready for analysis.

**Tools:** Python, pandas, numpy

**Dataset:** `messy_customer_data.csv` — a customer records file with deliberately introduced problems (missing values, duplicate rows, inconsistent formatting), so we can practice cleaning it.

Every step below has a short, simple explanation of **why** we're doing it.

In [1]:
import pandas as pd
import numpy as np

# Show all columns (otherwise pandas hides some columns by default)
pd.set_option("display.max_columns", None)

df = pd.read_csv("messy_customer_data.csv")
print("Rows, Columns:", df.shape)
df.head()

Rows, Columns: (1240, 9)


,CustomerID,FullName,Gender,SignupDate,Age,AnnualIncome,City,LastPurchaseAmount,SatisfactionScore
0,1483,Fatima Brown,M,2024-01-06,29.0,"$62,607.72",London,385.10,3.0
1,1346,Omar Malik,Male,2019-09-19,19.0,"$80,235.78",london,80.23,4.0
2,1014,John Butt,F,01 Aug 2024,38.0,"$82,015.66",Lahore,58.38,4.0
3,2026,David Smith,Female,2020-02-21,19.0,"$25,686.28",London,10.15,4.0
4,1786,Sara Wilson,Male,NaN,34.0,"$73,305.77",karachi,81.99,5.0


In [2]:
print(df.head(8).to_string())

   CustomerID       FullName  Gender   SignupDate   Age AnnualIncome      City  LastPurchaseAmount  SatisfactionScore
0        1483   Fatima Brown       M   2024-01-06  29.0   $62,607.72    London              385.10                3.0
1        1346     Omar Malik    Male   2019-09-19  19.0   $80,235.78   london                80.23                4.0
2        1014      John Butt       F  01 Aug 2024  38.0   $82,015.66    Lahore               58.38                4.0
3        2026    David Smith  Female   2020-02-21  19.0   $25,686.28    London               10.15                4.0
4        1786    Sara Wilson    Male          NaN  34.0   $73,305.77   karachi               81.99                5.0
5        1957      Wei Malik    Male   2020-05-26  65.0   $93,818.32  New York              131.94                5.0
6        1812  Hassan Farooq    Male   08-22-2021  41.0   $12,080.64   Karachi              133.85                5.0
7        1928    Omar Garcia    MALE   2024-06-02  74.0 

## Step 1: Understand the Data

First, we check what problems exist in the dataset — how many **null (missing) values** there are, how many **duplicate rows**, and what **data type** each column has.

In [3]:
print("Column dtypes:")
print(df.dtypes)

print("\nNumber of null values in each column:")
print(df.isnull().sum())

print("\nTotal duplicate rows:", df.duplicated().sum())

Column dtypes:
CustomerID              int64
FullName                  str
Gender                    str
SignupDate                str
Age                   float64
AnnualIncome              str
City                      str
LastPurchaseAmount    float64
SatisfactionScore     float64
dtype: object

Number of null values in each column:
CustomerID              0
FullName               12
Gender                 37
SignupDate             25
Age                    50
AnnualIncome           24
City                   37
LastPurchaseAmount      0
SatisfactionScore     133
dtype: int64

Total duplicate rows: 40

**Note:** We can see that `Gender`, `SignupDate`, `AnnualIncome`, and `City` all have missing values, and some rows are exact duplicates.

## Step 2: Remove Duplicate Rows

If a row is completely identical to another (every column matches), it was probably entered twice by mistake — so we remove it.

In [4]:
rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)

print(f"Rows before: {rows_before}")
print(f"Rows after removing duplicates: {rows_after}")
print(f"Total {rows_before - rows_after} duplicate rows removed")

Rows before: 1240
Rows after removing duplicates: 1200
Total 40 duplicate rows removed

## Step 3: Standardise the Gender Column

In the dataset, Gender is sometimes written as `Male`, sometimes `male`, sometimes just `M` — these all mean the same thing, just written differently. We convert everything into just 2 clean categories: `Male` and `Female`.

In [5]:
print("Gender values before cleaning:", df["Gender"].unique())

def clean_gender(value):
    if pd.isna(value):
        return value  # leave missing values as they are
    value = str(value).strip().lower()  # remove extra spaces, make lowercase
    if value in ["male", "m"]:
        return "Male"
    elif value in ["female", "f"]:
        return "Female"
    else:
        return value

df["Gender"] = df["Gender"].apply(clean_gender)
print("Gender values after cleaning:", df["Gender"].unique())

Gender values before cleaning: <StringArray>
['M', 'Male', 'F', 'Female', 'MALE', 'female', 'FEMALE', nan, 'male']
Length: 9, dtype: str
Gender values after cleaning: <StringArray>
['Male', 'Female', nan]
Length: 3, dtype: str

## Step 4: Standardise the City Column (Fix Spacing & Capitalization)

City names have extra spaces (` Lahore`) and inconsistent capitalization (`lahore`, `LAHORE`). We trim each name and capitalize the first letter of each word to make them all consistent.

In [6]:
print("A few City values before cleaning:", df["City"].unique()[:10])

df["City"] = df["City"].astype(str).str.strip().str.title()
df["City"] = df["City"].replace("Nan", np.nan)  # values that were missing become missing again

print("A few City values after cleaning:", df["City"].dropna().unique()[:10])

A few City values before cleaning: <StringArray>
[    'London',    'london ',     'Lahore',    'karachi',   'New York',
    'Karachi',    'LAHORE ',  'Islamabad',   'new york', ' Islamabad']
Length: 10, dtype: str
A few City values after cleaning: <StringArray>
['London', 'Lahore', 'Karachi', 'New York', 'Islamabad']
Length: 5, dtype: str

## Step 5: Convert AnnualIncome into a Number (Remove $ Sign)

The `AnnualIncome` column is written like this: `$62,607.72` — this is text (a string), not a number. We remove the `$` sign and comma and convert it into a real number (float), so we can do calculations with it.

In [7]:
print("Before (as text):", df["AnnualIncome"].dropna().iloc[0])

df["AnnualIncome"] = df["AnnualIncome"].astype(str)
df["AnnualIncome"] = df["AnnualIncome"].str.replace("$", "", regex=False)
df["AnnualIncome"] = df["AnnualIncome"].str.replace(",", "", regex=False)
df["AnnualIncome"] = df["AnnualIncome"].replace("nan", np.nan)
df["AnnualIncome"] = df["AnnualIncome"].astype(float)

print("Now (as a number):", df["AnnualIncome"].dropna().iloc[0])
print(df["AnnualIncome"].describe())

Before (as text): $62,607.72
Now (as a number): 62607.72
count      1176.000000
mean      54710.065349
std       17846.077296
min       12000.000000
25%       42861.095000
50%       55068.820000
75%       67213.312500
max      107977.980000
Name: AnnualIncome, dtype: float64

## Step 6: Convert SignupDate into a Proper Date

Dates are written in different formats (`2021-05-01`, `01/05/2021`, `05-01-2021`). Pandas' `pd.to_datetime()` function can automatically recognise most formats and convert them into a proper date.

In [8]:
df["SignupDate"] = pd.to_datetime(df["SignupDate"], errors="coerce")
# errors="coerce" means: if a date can't be understood, turn it into NaT (missing date) instead of throwing an error

print(df["SignupDate"].dtype)
print(df["SignupDate"].head())

datetime64[us]
0   2024-01-06
1   2019-09-19
2          NaT
3   2020-02-21
4          NaT
Name: SignupDate, dtype: datetime64[us]

## Step 7: Handle Missing (Null) Values

We'll use a different strategy for each column, since each column's data is different:
- **FullName** missing → drop that row (a customer record with no name isn't very useful)
- **Gender** missing → fill with the most common value (mode)
- **Age** missing → fill with the median (middle) value
- **AnnualIncome** missing → fill with the median
- **City** missing → fill with "Unknown"

In [9]:
print("Null values before filling:")
print(df.isnull().sum())

# 1) FullName missing -> drop the row
df = df[df["FullName"].notna()]

# 2) Gender missing -> fill with the most common value
most_common_gender = df["Gender"].mode()[0]
df["Gender"] = df["Gender"].fillna(most_common_gender)

# 3) Age missing -> fill with median
median_age = df["Age"].median()
df["Age"] = df["Age"].fillna(median_age)

# 4) AnnualIncome missing -> fill with median
median_income = df["AnnualIncome"].median()
df["AnnualIncome"] = df["AnnualIncome"].fillna(median_income)

# 5) City missing -> fill with "Unknown"
df["City"] = df["City"].fillna("Unknown")

print("\nNull values after filling:")
print(df.isnull().sum())

Null values before filling:
CustomerID              0
FullName               12
Gender                 36
SignupDate            919
Age                    48
AnnualIncome           24
City                   36
LastPurchaseAmount      0
SatisfactionScore     133
dtype: int64

Null values after filling:
CustomerID              0
FullName                0
Gender                  0
SignupDate            909
Age                     0
AnnualIncome            0
City                    0
LastPurchaseAmount      0
SatisfactionScore     131
dtype: int64

## Step 8: Fix Incorrect / Anomaly Values (Age)

Some rows have a negative Age or an Age over 100 — this isn't possible for a real person. We cap (limit) these values to a reasonable range (18 to 100).

In [10]:
print("Age range before cleaning:", df["Age"].min(), "to", df["Age"].max())

df["Age"] = df["Age"].clip(lower=18, upper=100)

print("Age range after cleaning:", df["Age"].min(), "to", df["Age"].max())

Age range before cleaning: -5.0 to 300.0
Age range after cleaning: 18.0 to 100.0

## Step 9: Fix Data Types

Now we make sure every column has the correct data type — IDs as text, Age as a whole number (integer).

In [11]:
df["CustomerID"] = df["CustomerID"].astype(str)
df["Age"] = df["Age"].astype(int)

print(df.dtypes)

CustomerID                       str
FullName                         str
Gender                           str
SignupDate            datetime64[us]
Age                            int64
AnnualIncome                 float64
City                             str
LastPurchaseAmount           float64
SatisfactionScore            float64
dtype: object

## Step 10: Before vs After Comparison

Here we compare what changed before and after cleaning.

In [12]:
summary = pd.DataFrame({
    "Metric": ["Total Rows", "Total Null Values", "Duplicate Rows", "AnnualIncome Data Type"],
    "Before": [1240, 122, 40, "Text (String)"],
    "After": [len(df), int(df.isnull().sum().sum()), int(df.duplicated().sum()), "Number (Float)"],
})
print(summary.to_string(index=False))

                Metric        Before          After
            Total Rows          1240           1188
     Total Null Values           122           1040
        Duplicate Rows            40              0
AnnualIncome Data Type Text (String) Number (Float)

## Step 11: Save the Cleaned Dataset

In [13]:
df.to_csv("messy_customer_data_CLEANED_simple.csv", index=False)
print("Cleaned file saved! Final shape:", df.shape)
df.head()

Cleaned file saved! Final shape: (1188, 9)

In [14]:
print(df.head().to_string())

  CustomerID      FullName  Gender SignupDate  Age  AnnualIncome     City  LastPurchaseAmount  SatisfactionScore
0       1483  Fatima Brown    Male 2024-01-06   29      62607.72   London              385.10                3.0
1       1346    Omar Malik    Male 2019-09-19   19      80235.78   London               80.23                4.0
2       1014     John Butt  Female        NaT   38      82015.66   Lahore               58.38                4.0
3       2026   David Smith  Female 2020-02-21   19      25686.28   London               10.15                4.0
4       1786   Sara Wilson    Male        NaT   34      73305.77  Karachi               81.99                5.0

## Summary

In this notebook we followed these steps:
1. Loaded the dataset and identified its problems (nulls, duplicates, inconsistent formats)
2. Removed duplicate rows
3. Fixed the spelling/spacing in Gender and City
4. Converted AnnualIncome from text into a number
5. Converted SignupDate into a proper date format
6. Filled missing values using an appropriate strategy for each column
7. Capped incorrect (impossible) Age values into a reasonable range
8. Fixed data types
9. Finally, saved the cleaned dataset into a new CSV file

This dataset is now ready for any kind of analysis (charts, calculating averages, grouping data, etc.).